In [1]:
import numpy as np
import cv2 as cv

In [5]:
# RGB Histogram

src = cv.imread("images/apple2.jpg", cv.IMREAD_COLOR)

bgr_planes = cv.split(src)

histSize = 256
histRange = (0, 255)

b_hist = cv.calcHist(bgr_planes, [0], None, [histSize], histRange)
g_hist = cv.calcHist(bgr_planes, [1], None, [histSize], histRange)
r_hist = cv.calcHist(bgr_planes, [2], None, [histSize], histRange)

hist_w = 256*3
hist_h = 400
histImage = np.zeros((hist_h, hist_w, 3), dtype=np.uint8)

cv.normalize(b_hist, b_hist, alpha=0, beta=hist_h, norm_type=cv.NORM_MINMAX)
cv.normalize(g_hist, g_hist, alpha=0, beta=hist_h, norm_type=cv.NORM_MINMAX)
cv.normalize(r_hist, r_hist, alpha=0, beta=hist_h, norm_type=cv.NORM_MINMAX)

for i in range(0, histSize):
    # histImage에 시작점, 끝점,색상, 두께 
    cv.line(histImage, (i,       hist_h - int(np.round(b_hist[i]))),   (i, hist_h-0),     (255, 0, 0), thickness=2)
    cv.line(histImage, (i + 256, hist_h - int(np.round(g_hist[i-1]))), (i+256, hist_h-0), (0, 255, 0), thickness=2)
    cv.line(histImage, (i + 512, hist_h - int(np.round(r_hist[i]-1))), (i+512, hist_h-0), (0, 0, 255), thickness=2)

cv.imshow('Source Image', src)
cv.imshow('Histogram', histImage)
cv.waitKey(0)
cv.destroyAllWindows()

In [9]:
# GrayScale Histogram

src = cv.imread("images/cat on laptop.jpg", cv.IMREAD_GRAYSCALE)

histSize = 256
histRange = (0, 255)

gray_hist = cv.calcHist([src], [0], None, [histSize], histRange)

hist_w = 256
hist_h = 400
histImage = np.zeros((hist_h, hist_w, 1), dtype=np.uint8)

cv.normalize(gray_hist, gray_hist, alpha=0, beta=hist_h, norm_type=cv.NORM_MINMAX)

for i in range(0, histSize):
    # histImage에 시작점, 끝점,색상, 두께 
    cv.line(histImage, (i,       hist_h - int(np.round(gray_hist[i]))),   (i, hist_h-0),     (255, 0, 0), thickness=2)

cv.imshow('Source Image', src)
cv.imshow('Histogram', histImage)
cv.waitKey(0)
cv.destroyAllWindows()


In [4]:
# Histogram Equalization

def draw_histogram(img):
    histSize = 256
    histRange = (0, 255)
    
    hist_w = 256
    hist_h = img.shape[0]
    histImage = np.zeros((hist_h, hist_w, 1), dtype=np.uint8)

    gray_hist = cv.calcHist([img], [0], None, [histSize], histRange)
    
    cv.normalize(gray_hist, gray_hist, alpha=0, beta=hist_h, norm_type=cv.NORM_MINMAX)
    
    for i in range(0, histSize):
        # histImage에 시작점, 끝점,색상, 두께 
        cv.line(histImage, (i, hist_h - int(np.round(gray_hist[i]))), (i, hist_h-0), (255, 0, 0), thickness=2)

    return histImage

img_gray = cv.imread("images/test8.png", cv.IMREAD_GRAYSCALE)

img_hist = draw_histogram(img_gray)
result1 = cv.hconcat((img_gray, img_hist))
cv.imshow('Original', result1)

img_equ = cv.equalizeHist(img_gray)
img_hist = draw_histogram(img_equ)
result2 = cv.hconcat((img_equ, img_hist))
cv.imshow('Result', result2)
cv.waitKey(0)
cv.destroyAllWindows()

In [17]:
# CLAHE (Contrast Limited Adaptive Histogram Equalization)

img_gray = cv.imread("images/test11.png", cv.IMREAD_GRAYSCALE)

img_hist = draw_histogram(img_gray)
result1 = cv.hconcat((img_gray, img_hist))
cv.imshow('Original', result1)

clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

img_clahe = clahe.apply(img_gray)

img_hist = draw_histogram(img_clahe)
result2 = cv.hconcat((img_clahe, img_hist))
cv.imshow('Result', result2)
cv.waitKey(0)
cv.destroyAllWindows()

In [19]:
# Template Matching - 1 Object

img_color = cv.imread("images/test7.jpg", cv.IMREAD_COLOR)
img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

img_template = cv.imread("images/template.jpg", cv.IMREAD_GRAYSCALE)

w, h = img_template.shape[:2]

res = cv.matchTemplate(img_gray, img_template, cv.TM_CCOEFF_NORMED)

min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)

top_left = max_loc
bottom_right = (top_left[0] + w, top_left[1] + h)

cv.rectangle(img_color, top_left, bottom_right, (0, 0, 255), 2)

cv.imshow('Template', img_template)
cv.imshow('Result', img_color)
cv.waitKey(0)
cv.destroyAllWindows()

In [20]:
# Template Matching - Multiple Object

detectedObjects = []

def notInList(newObject):
    for detectedObject in detectedObjects:
        a = newObject[0] - detectedObject[0]
        b = newObject[1] - detectedObject[1]
        if np.sqrt(a*a + b*b) < 5 : 
            return False
    return True

img_color = cv.imread("images/test7.jpg", cv.IMREAD_COLOR)
img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

img_template = cv.imread("images/template.jpg", cv.IMREAD_GRAYSCALE)
w, h = img_template.shape[:2]

res = cv.matchTemplate(img_gray, img_template, cv.TM_CCOEFF_NORMED)

count = 0
for x in range(res.shape[1]):
    for y in range(res.shape[0]):
        if res[y, x] > 0.9 and notInList((x, y)):
            detectedObjects.append((x, y))
            top_left = (x, y)
            bottom_right = (top_left[0] + w, top_left[1] + h)
            cv.rectangle(img_color, top_left, bottom_right, (0, 0, 255), 2)
            count += 1

print(count)

cv.imshow('Template', img_template)
cv.imshow('Result', img_color)
cv.waitKey(0)
cv.destroyAllWindows()

11


In [22]:
# Contour

img_color = cv.imread("images/test12.jpg", cv.IMREAD_COLOR)
img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

cv.imshow('Gray Original', img_gray)
cv.waitKey(0)

img_edge = cv.Canny(img_gray, 50, 150)
cv.imshow('Edge Detect', img_edge)
cv.waitKey(0)

img_edge = cv.bitwise_not(img_edge)
cv.imshow('Inverse Edge Detect', img_edge)
cv.waitKey(0)

contours = cv.findContours(img_edge.copy(), cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
cv.drawContours(img_edge, contours[0], -1, (0, 0, 0), 1)
cv.imshow('Draw Contours', img_edge)
cv.waitKey(0)

nlabels, labels, stats, centroids = cv.connectedComponentsWithStats(img_edge)

for i in range(nlabels):
    if i<2:
        continue

    area = stats[i, cv.CC_STAT_AREA]
    center_x, center_y = int(centroids[i, 0]), int(centroids[i, 1])
    left, top = stats[i, cv.CC_STAT_LEFT], stats[i, cv.CC_STAT_TOP]
    width, height = stats[i, cv.CC_STAT_WIDTH], stats[i, cv.CC_STAT_HEIGHT]

    top_left = (left, top)
    bottom_right = (left + width, top + height)
    center_loc = (center_x, center_y)
    text_loc = (left + 20, top + 20)

    if area > 50:
        cv.rectangle(img_color, top_left, bottom_right, (0, 0, 255), 1)
        cv.circle(img_color, center_loc, 5, (255, 0, 0), 1)
        cv.putText(img_color, str(i), text_loc, cv.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 3)

cv.imshow('Result', img_color)
cv.waitKey(0)
cv.destroyAllWindows()

In [38]:
# Contour - 2 Object in Different Color

img_color = cv.imread("images/test13.jpg", cv.IMREAD_COLOR)
img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

ret, img_binary = cv.threshold(img_gray, 150, 255, cv.THRESH_BINARY_INV)

kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
img_binary = cv.morphologyEx(img_binary, cv.MORPH_CLOSE, kernel)

contours, hierarchy = cv.findContours(img_binary, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
cv.drawContours(img_color, contours, 0, (0, 0, 255), 3)
cv.drawContours(img_color, contours, 1, (0, 255, 0), 3)
# print(contours)

cv.imshow('Draw Contours', img_color)
cv.waitKey(0)
cv.destroyAllWindows()

In [39]:
# Calcuate Contour Area

img_color = cv.imread("images/test13.jpg", cv.IMREAD_COLOR)
img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

ret, img_binary = cv.threshold(img_gray, 150, 255, cv.THRESH_BINARY_INV)

kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
img_binary = cv.morphologyEx(img_binary, cv.MORPH_CLOSE, kernel)

contours, hierarchy = cv.findContours(img_binary, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)

cv.drawContours(img_color, contours, 0, (0, 0, 255), 3)
cv.drawContours(img_color, contours, 1, (0, 255, 0), 3)
# print(contours)

for i, contour in enumerate(contours):
    area = cv.contourArea(contour)
    print(i, '-', area)

cv.imshow('Draw Contours', img_color)
cv.waitKey(0)
cv.destroyAllWindows()

0 - 24072.5
1 - 47240.5


In [24]:
# Contour in Line

img_color = cv.imread("images/square.png", cv.IMREAD_COLOR)
cv.imshow('Original', img_color)
cv.waitKey(0)

img_gray = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

ret, img_binary = cv.threshold(img_gray, 150, 255, cv.THRESH_BINARY_INV)

kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
img_binary = cv.morphologyEx(img_binary, cv.MORPH_CLOSE, kernel)
cv.imshow('Close Mocrphology', img_binary)
cv.waitKey(0)

contours, hierarchy = cv.findContours(img_binary, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)

cv.drawContours(img_color, contours, 0, (0, 0, 255), 3)

cv.imshow('Draw Contours', img_color)
cv.waitKey(0)


if len(contours) != 1:
    contours_list = sorted(contours, key=cv.contourArea, reverse=True)
    contours = [contours_list[1]]

for contour in contours:
    epsilon = 0.03 * cv.arcLength(contour, True)
    approx = cv.approxPolyDP(contour, epsilon, True)
    cv.drawContours(img_color, [approx], 0, (0, 255, 0), 3)
    # print(approx)

cv.imshow('Draw Line Contours', img_color)
cv.waitKey(0)
cv.destroyAllWindows()